In [ ]:
import sys
from pathlib import Path

# Bootstrap: find the project's src/ directory dynamically by walking up from the
# current working directory until a folder containing a .env file is found, then add
# its src/ to the path. This makes the notebook portable across machines (Windows/Mac)
# with no hardcoded absolute paths.
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

# Project-wide config (PROCESSED_DIR, DOWNLOADS_DIR, BROWSER_HEADERS, etc.) and the
# download-log helpers, plus standard libraries used throughout this notebook.
from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import glob, os, re, warnings
import pycountry

# Load the download log (tracks per-source currency/filenames) into memory.
log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## AREAER FARI Pipeline (capital-account restrictiveness)

**Source:** IMF Annual Report on Exchange Arrangements and Exchange Restrictions (AREAER) — Indices tab, FARI (Financial Account Restrictiveness Index)
**Access:** **MANUAL** download. The portal (elibrary-areaer.imf.org) is WAF-blocked to programmatic access and JS-gated, so files are exported by hand from the Indices tab and saved to Downloads.
**Concept role:** PRIMARY tier-1 — capital-account openness (de jure). IMF-native authoritative source.

### Fields produced
- `fari_aggregate` (0–1, higher = MORE restrictive): overall capital-account restrictiveness across 56 transaction categories. **PRIMARY scored field.**
- `fari_fdi_aggregate` (0–1): FDI-specific restrictiveness. **PRIMARY scored field** (FDI openness is highly investment-relevant and not separable in derivative indices like Chinn-Ito).
- `fari_inflow` / `fari_outflow` / `fari_fdi_inflow` / `fari_fdi_outflow`: directional splits, retained as supplementary detail.

### MANUAL SNAPSHOT — does not auto-update
To refresh: portal → **Indices** tab → select **FARI Aggregate** + **FARI - FDI** family, all countries, **annual**, full range → download to Downloads. This notebook auto-detects the latest `FARIReportByCountry*.xlsx`. See `docs/instructions_data_maintenance.md` for the full procedure. **MANUAL UPDATE point.**

### Notes
- 194 countries, 1999–2024. Coverage near-universal (strongest tier).
- Latest year (2024) is **partial** per source footnote ("based on each country's position date").
- Direction validated: Hong Kong ~0.02 / Singapore ~0.05 (open); Bangladesh ~0.77 (closed).
- Complemented by **Chinn-Ito** (KAOPEN, automated derivative) and **Reinhart-Rogoff** (de facto exchange-rate regime).
- Companion **ACI** file (AREAER Change Index) is downloaded but **not built** into the score — ACI measures policy *changes* (direction of travel), a different construct from FARI's *level*; deferred as optional supplementary.

> Diagnostic cells, if ever added during debugging, must be marked
> `# ⚠️ DIAGNOSTIC — DELETE THIS CELL BEFORE COMMITTING` and removed before commit.
> This committed version contains no diagnostic cells.


In [ ]:
# Locate the most recent FARI Index Report in the Downloads folder. The file is exported
# manually from the AREAER portal and saved with a download-date suffix
# (e.g. FARIReportByCountry_06_18_2026_112959.xlsx); the prefix match ignores that suffix
# and os.path.getmtime picks the newest, so repeated re-exports are handled automatically.
fari_files = glob.glob(os.path.join(DOWNLOADS_DIR, "FARIReportByCountry*.xlsx"))
if not fari_files:
    raise FileNotFoundError(
        "No FARIReportByCountry*.xlsx in Downloads. Export FARI from the AREAER portal "
        "Indices tab (see docs/instructions_data_maintenance.md)."
    )
fari_path = max(fari_files, key=os.path.getmtime)
print(f"Using: {os.path.basename(fari_path)}")


In [ ]:
# Silence openpyxl's cosmetic "no default style" warning on these IMF exports.
warnings.simplefilter("ignore")

# The export has two metadata rows (title, blank) above the real header on row index 2,
# so read with header=2. Layout is WIDE: Index Name | IFS Code | Country | <one col per year-end date>.
fari = pd.read_excel(fari_path, header=2)

# Keep only the six genuine FARI index rows; this drops the interleaved footnote/metadata
# rows at the bottom of the sheet (which carry text in Index Name and NaN elsewhere).
VALID_INDICES = ['FARI Aggregate', 'FARI Inflow', 'FARI Outflow',
                 'FARI - FDI Aggregate', 'FARI - FDI Inflow', 'FARI - FDI Outflow']
fari = fari[fari['Index Name'].isin(VALID_INDICES)].copy()

# Identify the year columns (headers are dates like '12/31/1999') and map each to its
# integer year, parsed FROM the header text — no hardcoded year list, so new annual
# columns are picked up automatically on future exports.
id_cols = ['Index Name', 'IFS Code', 'Country']
year_cols = [c for c in fari.columns if c not in id_cols]
col_to_year = {}
for c in year_cols:
    m = re.search(r'(\d{4})', str(c))
    if m:
        col_to_year[c] = int(m.group(1))

# Reshape wide -> long: one row per (index, country, year).
long = fari.melt(id_vars=id_cols, value_vars=list(col_to_year.keys()),
                 var_name='year_col', value_name='value')
long['year'] = long['year_col'].map(col_to_year)
long['value'] = pd.to_numeric(long['value'], errors='coerce')  # 'nan' strings -> real NaN
long = long.drop(columns='year_col').dropna(subset=['value'])    # drop missing country-years

# Pivot so each of the six index variants becomes its own column, keyed by country-year.
wide = long.pivot_table(index=['IFS Code', 'Country', 'year'],
                        columns='Index Name', values='value').reset_index()
wide.columns.name = None

# Rename source index labels to clean snake_case field names.
rename_map = {
    'FARI Aggregate': 'fari_aggregate',           # PRIMARY: overall capital-account restrictiveness
    'FARI Inflow': 'fari_inflow',
    'FARI Outflow': 'fari_outflow',
    'FARI - FDI Aggregate': 'fari_fdi_aggregate', # PRIMARY: FDI-specific restrictiveness
    'FARI - FDI Inflow': 'fari_fdi_inflow',
    'FARI - FDI Outflow': 'fari_fdi_outflow',
}
wide = wide.rename(columns=rename_map)

print(f"Reshaped: {wide.shape}  | countries: {wide['Country'].nunique()}  "
      f"| years: {wide['year'].min()}-{wide['year'].max()}")


In [ ]:
# Map country names to ISO3. The file identifies countries by IMF IFS code + name; the
# framework keys on ISO3. pycountry handles standard names; the dict below covers the IMF's
# non-standard spellings and edge cases. NOTE: 'Cote d Ivoire' here uses the CURLY apostrophe
# the IMF file actually contains. MANUAL UPDATE: if the unmapped list below is non-empty on a
# future export, add the new name(s) here.
MANUAL_ISO3 = {
    'China, P.R.: Mainland': 'CHN', 'China, P.R.: Hong Kong': 'HKG', 'Hong Kong SAR': 'HKG',
    'China, P.R.: Macao': 'MAC', 'Macao SAR': 'MAC', 'Korea, Republic of': 'KOR',
    "Korea, Dem. People's Rep. of": 'PRK', 'Taiwan Province of China': 'TWN',
    'Russian Federation': 'RUS', 'Iran, Islamic Republic of': 'IRN', 'Iran, I.R. of': 'IRN',
    'Venezuela, Republica Bolivariana de': 'VEN', 'Venezuela, Rep. Bolivariana de': 'VEN',
    'Bolivia': 'BOL', 'Tanzania': 'TZA', 'Vietnam': 'VNM', "Lao People's Dem. Rep": 'LAO',
    'Lao P.D.R.': 'LAO', 'Syrian Arab Republic': 'SYR', 'Moldova': 'MDA',
    'North Macedonia': 'MKD', 'Czech Republic': 'CZE', 'Slovak Republic': 'SVK',
    'Kyrgyz Republic': 'KGZ', 'Brunei Darussalam': 'BRN', 'Cabo Verde': 'CPV',
    'Cape Verde': 'CPV', 'Congo, Dem. Rep. of the': 'COD', 'Congo, Republic of': 'COG',
    'Congo, Rep. of': 'COG', 'Gambia, The': 'GMB', 'Bahamas, The': 'BHS',
    'Micronesia, Fed. States of': 'FSM', 'Micronesia, Federated States of': 'FSM',
    'Sao Tome and Principe': 'STP', 'Timor-Leste': 'TLS',
    'Eswatini': 'SWZ', 'Kosovo': 'XKX', 'Myanmar (Burma)': 'MMR', 'Myanmar': 'MMR',
    'South Sudan': 'SSD', 'Curacao and Sint Maarten': 'CUW', 'Aruba': 'ABW',
    'Turkey': 'TUR', 'West Bank and Gaza': 'PSE',
    'Yemen, Republic of': 'YEM', 'Yemen, Rep. of': 'YEM',
    'Congo, Democratic Republic of the': 'COD',
    'Korea': 'KOR', 'Micronesia': 'FSM', 'North Macedonia, Republic of': 'MKD',
    'Russia': 'RUS', 'St. Kitts and Nevis': 'KNA', 'St. Lucia': 'LCA',
    'St. Vincent and the Grenadines': 'VCT',
}
# Two names in the IMF file use a curly apostrophe; add them programmatically so this
# source cell stays ASCII-clean and the keys still match the file's exact bytes.
MANUAL_ISO3['C\u00f4te d\u2019Ivoire'] = 'CIV'   # curly-apostrophe variant in the file
MANUAL_ISO3["Cote d'Ivoire"] = 'CIV'             # straight-apostrophe fallback
MANUAL_ISO3['S\u00e3o Tom\u00e9 and Pr\u00edncipe'] = 'STP'

def to_iso3(name):
    # Manual dict first (handles IMF quirks), then pycountry fuzzy lookup; None if unresolved.
    n = str(name).strip()
    if n in MANUAL_ISO3:
        return MANUAL_ISO3[n]
    try:
        return pycountry.countries.lookup(n).alpha_3
    except LookupError:
        return None

wide['country_code'] = wide['Country'].map(to_iso3)

# Surface any unmapped names so they can be added to MANUAL_ISO3 (MANUAL UPDATE point).
unmapped = sorted(wide[wide['country_code'].isna()]['Country'].unique())
print(f"Unmapped (add to MANUAL_ISO3 if any): {unmapped}")
print(f"Mapped: {wide['country_code'].notna().sum()} of {len(wide)} rows, "
      f"{wide[wide['country_code'].notna()]['country_code'].nunique()} countries")


In [ ]:
# Assemble the final tidy panel: keys first, then the two PRIMARY aggregate fields, then
# the supplementary directional splits. Keep the source country name for traceability.
final = wide[['country_code', 'Country', 'year',
              'fari_aggregate', 'fari_fdi_aggregate',
              'fari_inflow', 'fari_outflow', 'fari_fdi_inflow', 'fari_fdi_outflow']].copy()
final = final.rename(columns={'Country': 'country_name_source'})
final = final.sort_values(['country_code', 'year']).reset_index(drop=True)

# Write the processed output.
output_path = os.path.join(PROCESSED_DIR, "areaer_fari_clean.csv")
final.to_csv(output_path, index=False)
print(f"Written: {output_path}  | shape: {final.shape}")

# Data currency is DERIVED from the data (latest year present), never hardcoded.
latest_year = int(final['year'].max())
retrieval_date = datetime.today().strftime("%Y-%m-%d")

# Record the manual download in the log.
update_entry(
    "IMF_AREAER",
    last_successful_download_date=retrieval_date,
    data_as_of_date=f"{latest_year} (AREAER {latest_year}; {latest_year} partial per source)",
    local_filename="areaer_fari_clean.csv",
    latest_available_version=f"AREAER {latest_year} (FARI Indices export)",
    notes=("IMF AREAER FARI (capital-account restrictiveness, de jure). PRIMARY tier-1. MANUAL: "
           "portal WAF-blocked + JS-gated; FARI exported by hand to Downloads, pipeline auto-detects "
           "latest FARIReportByCountry*.xlsx. MANUAL SNAPSHOT. fari_aggregate + fari_fdi_aggregate "
           "(0-1, higher=more restrictive) primary; inflow/outflow supplementary. 194 countries, "
           f"1999-{latest_year} ({latest_year} partial). Direction validated (HK 0.02/SGP 0.05 open; "
           "BGD 0.77 closed). Complemented by Chinn-Ito (automated) + Reinhart-Rogoff (de facto regime). "
           "ACI downloaded but not built (policy-change construct; deferred).")
)
print_entry("IMF_AREAER")


In [ ]:
# Update the source registry (access method + approach) so the access pattern is tracked
# alongside every other source. Update-in-place if the row exists, else append.
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

areaer_notes = (
    "IMF AREAER FARI capital-account restrictiveness (de jure). PRIMARY tier-1. MANUAL: portal "
    "WAF-blocked + JS-gated; export FARI Indices by hand (Indices tab -> FARI Aggregate + FARI-FDI "
    "families, all countries, annual, full range) to Downloads; pipeline auto-detects latest "
    "FARIReportByCountry*.xlsx. fari_aggregate + fari_fdi_aggregate (0-1, higher=more restrictive) "
    "primary; inflow/outflow supplementary. 194 countries 1999-2024 (2024 partial). Complemented by "
    "Chinn-Ito (automated) + Reinhart-Rogoff (de facto regime). ACI downloaded, not built. IFS-code "
    "file, ISO3 mapped via name + MANUAL_ISO3 dict (needs pycountry)."
)
approach = ("manual export to Downloads (portal WAF-blocked); auto-detect latest "
            "FARIReportByCountry*.xlsx, drop footnotes, reshape wide->long->pivot, ISO3 via name")

if (registry_df['source_id'] == 'IMF_AREAER').any():
    registry_df.loc[registry_df['source_id'] == 'IMF_AREAER', 'access_method'] = 'manual_download'
    registry_df.loc[registry_df['source_id'] == 'IMF_AREAER', 'python_approach'] = approach
    registry_df.loc[registry_df['source_id'] == 'IMF_AREAER', 'notes'] = areaer_notes
    print("Updated IMF_AREAER registry row")
else:
    registry_df = pd.concat([registry_df, pd.DataFrame([{
        'source_id': 'IMF_AREAER', 'access_method': 'manual_download',
        'python_approach': approach, 'notes': areaer_notes}])], ignore_index=True)
    print("Added IMF_AREAER registry row")

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print(registry_df[registry_df['source_id'] == 'IMF_AREAER'][['source_id', 'access_method']].to_string())
